# v9 node dial — geometry, performance, and an honest choice of alpha

The dial moves the graph's **input**:

$$x_{\text{prot}}(\alpha)\;=\;\mu\;+\;\alpha\cdot\text{centred}(\text{ESM})\;+\;(1-\alpha)\cdot\text{centred}(\text{identity vector})$$

one fixed near-orthogonal random vector per receptor, drawn once by name. So

| alpha | what the receptor node is | the model it equals |
|---|---|---|
| **0** | identity alone — who this receptor is, nothing about what it looks like | the one-hot graph |
| **1** | the embedding file itself, unchanged | the graph as it was before the dial |

**The direction is the OPPOSITE of the v8 gate**, where alpha ran structure → function.
Never put the two in one frame: `nodes` separates them and the loader defaults keep them
apart.

Because alpha=1 *is* the pre-dial graph — `rho=1` short-circuits to the embedding dict,
the gate is off, the nodes are not swapped — the old `graph_legacy` arm is no longer run
alongside it. On a seeded run the two are one computation, and the second copy of it was
a sixth of the graph budget spent to reprint a column. Nothing below draws it.

### The three questions, in order

1. **Geometry** — does the receptor cloud actually travel between the two ideological
   ends? ESM on one side, the train response profile on the other, three second-order
   measures, each against its own permutation null.
2. **Performance** — what the metrics and the mean places do along the dial.
3. **The choice** — which alpha to report. Made on **validation**, then and only then
   carried to test.

### Why step 3 is built the way it is

Alpha is a hyperparameter: fixed before training, used unchanged at inference. Choosing
it by the score on the folds the paper reports is selection on the test set — the
winner's margin is then partly the luck of those folds, and quoting it as if alpha had
been fixed in advance overstates the claim.

A run made by the current sweep scores **train, val and test** from one computation and
writes `val_metrics_*.csv` beside the test file — nothing extra to do.

A run made **before that** (`v9_node_dial`, and anything older) wrote test only.
`scripts/analysis/val_rescore.py` fills the gap without retraining anything: it reloads
each cell's dumped receptor cloud, refits the boosting head on the same train rows with
the same seed — reproducing the estimator whose test score is already on disk — and asks
it for the validation rows instead:

```sh
python scripts/analysis/val_rescore.py --root results/graph/v9_node_dial --dry-run
python scripts/analysis/val_rescore.py --root results/graph/v9_node_dial
```

`ROOT_DIR` below selects which run is read. The two are **different series** — one draws
the graph's initialisation from the global RNG, the other seeds it — so read each inside
itself and never quote a number from one against a number from the other.

Everything below is drawn from `scripts/analysis/alpha_grid.py` and
`scripts/analysis/alpha_choice.py`. **This notebook computes nothing of its own** — if a
number here disagrees with the CLI, the notebook is the one that is wrong.

In [ ]:
# ------------------------------------------------------------------ KNOBS ---
ROOT_DIR   = "results/graph/v9_seeded"      # the sweep directory
MOL_SOURCE = "chemberta"                    # chemberta | gin | ecfp | None = all
DATASET    = None                           # None = every dataset on disk
REGIME     = None                           # None = both regimes
SEED       = None                           # None = every model seed present
CI_LEVEL   = 0.95                           # Student-t, over the splits
GEOM_SCALE = "value"                        # "value" = the measure, "z" = vs its null
METRICS    = "headline"                     # "headline" (5 on regression, 4 on the
                                            # binary pool) | "all" (12 on regression --
                                            # readable only a few at a time) | a list
SHOW_PLACES_FOR = None                      # None = whatever METRICS resolves to
PLACE_ALPHAS = None                         # dial positions 2.5 puts in the matrix;
                                            # None = every one that ran
COLS_SERIES = 2                             # per-series panels per row
COLS_METRIC = 3                             # per-metric panels per row
SAVE_FIGS  = False
FIG_DIR    = "results/graph/v9_seeded/figures"

NODES = "nodedial"        # do NOT widen this: alpha means the opposite on the v8 gate

In [ ]:
import pathlib, sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

_root = pathlib.Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from scripts.analysis import alpha_grid as ag
from scripts.analysis import alpha_choice as ac

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 60)

KW = dict(root=ROOT_DIR, mol_source=MOL_SOURCE, nodes=NODES,
          dataset=DATASET, regime=REGIME, seed=SEED)

test = ag.load(split="test", **KW)
try:
    val = ag.load(split="val", **KW)
except SystemExit:
    val = None

MET = ag.metric_for(test)                       # metric of record per dataset
SERIES = sorted(test.series.unique())
print(f"{len(SERIES)} series, metric of record {MET}")
if val is None:
    print("\n!! NO VALIDATION FRAME -- block 3 will not run.\n"
          f"   python scripts/analysis/val_rescore.py --root {ROOT_DIR} --dry-run\n"
          f"   python scripts/analysis/val_rescore.py --root {ROOT_DIR}")
ag.coverage(test)

## How small a difference can this grid resolve?

One number before any curve is read: **the repeat floor**, the ruler every gap along the
dial is held against. It is **not** the across-fold interval — that one is 2–30× wider,
shared by every row, and largely cancels between two of them.

It is measured from the model seeds: the spread across seeds *inside* a fold, pooled over
folds, divided by `sqrt(seeds)` because every curve below is a mean over exactly those
seeds. That is the model noise still left in a fold mean, and a difference smaller than it
is not a difference this grid can see.

A single-seed run has no such spread to measure, so it has no floor and the panels simply
draw none. (The historical stand-in — the gap between alpha=1 and a separately trained
`graph_legacy`, two draws of one construction — is gone with the arm; on a seeded run it
would read zero anyway.)

In [ ]:
# The floor: model noise left in a fold MEAN, which is what every curve is made of.
# Measured from the seed spread inside each fold -- so a run with one seed has none.
FLOOR, SEEDS = {}, {}
for s in SERIES:
    sub = test[test.series == s]
    ns = ag.noise_split(sub, MET[sub.dataset.iloc[0]])
    f = ns["floor"].dropna() if len(ns) else []
    if len(f):
        FLOOR[s] = float(f.mean())
        SEEDS[s] = float(ns["n_seeds"].mean())

if FLOOR:
    print("repeat floor per series, on the metric of record "
          "(model SD inside a fold / sqrt(seeds)):")
    for s, v in FLOOR.items():
        print(f"  {s:<34} {v:.4f}   [{SEEDS[s]:.0f} seeds]")
    print(f"  {'POOLED':<34} {np.mean(list(FLOOR.values())):.4f}"
          "   <-- the size a difference must clear to be readable")
else:
    print("one model seed per cell -- no floor to measure, and the panels draw "
          "none." + chr(10) + "Rerun the sweep with --seeds 42 43 44 45 46 "
          "to get one.")

In [ ]:
# ------------------------------------------------------- plotting helpers ---
C = {"esm": "#1f6fb4", "fun": "#c9531f", "dial": "#1f6fb4", "val": "#c9531f",
     "boost_full": "#6b7280", "naive": "#b9bec6",
     "ink": "#1f2328", "muted": "#6b7280", "grid": "#e8e8e4"}
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white",
                     "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.edgecolor": "#d7d7d2"})

def panels(nrow, ncol, w=3.5, h=2.7, **kw):
    fig, ax = plt.subplots(nrow, ncol, figsize=(w * ncol, h * nrow), squeeze=False, **kw)
    return fig, ax

def grid(n, cols, w=3.5, h=2.7, **kw):
    """`n` panels wrapped at `cols` per row, spares switched off.

    A row of twelve panels is a row nobody reads -- and the regression battery IS
    twelve wide. Wrapping trades width, which the eye and the screen are short of,
    for height, which a notebook has for free.

    Yields (ax, bottom, left) per panel: once the last row is short, "is this the
    bottom of its column" and "is this the left edge" stop being i//cols arithmetic
    the caller should be redoing, and they are what the axis labels hang on.
    """
    cols = max(1, min(int(cols), n))
    rows = -(-n // cols)
    fig, AX = plt.subplots(rows, cols, figsize=(w * cols, h * rows), squeeze=False, **kw)
    flat = [a for r in AX for a in r]
    for a in flat[n:]:
        a.axis("off")
    return fig, [(a, i + cols >= n, i % cols == 0) for i, a in enumerate(flat[:n])]

def metric_list(df, ds):
    """What METRICS asks for, resolved against what this dataset actually wrote."""
    if isinstance(METRICS, (list, tuple)):
        have = ag.metrics_available(df, ds)
        return [m for m in METRICS if m in have]
    return ag.metrics_available(df, ds, which=METRICS or "all")

def band(ax, d, color, label=None, lw=2):
    """A curve with its across-fold interval. The interval is context, not the test:
    every row shares the folds, so overlapping bands do NOT mean 'no difference'."""
    if d.empty:
        return
    d = d.sort_values("alpha")
    x = d["alpha"].to_numpy(float)
    ax.fill_between(x, d["lo"], d["hi"], color=color, alpha=0.10, lw=0)
    ax.plot(x, d["mean"], color=color, lw=lw, label=label, zorder=3)
    ax.plot(x, d["mean"], "o", color=color, ms=4, mec="white", mew=1, zorder=4)

def floor_bar(ax, series, x=-0.09):
    """The repeat gap as an I-beam: the smallest difference this cell can resolve."""
    g = FLOOR.get(series)
    if not g:
        return
    y0, y1 = ax.get_ylim()
    mid = y0 + 0.42 * (y1 - y0)
    ax.plot([x, x], [mid - g / 2, mid + g / 2], color=C["ink"], lw=1.6, clip_on=False)
    for yy in (mid - g / 2, mid + g / 2):
        ax.plot([x - 0.015, x + 0.015], [yy, yy], color=C["ink"], lw=1.6, clip_on=False)
    ax.annotate(f"repeat\n{g:.3f}", xy=(x, mid + g / 2), xytext=(0, 5),
                textcoords="offset points", fontsize=7, ha="center", va="bottom",
                color=C["ink"], annotation_clip=False, linespacing=1.15)
    ax.set_xlim(x - 0.05, 1.0)

def ref_line(ax, v, color, text, xs=None, ys=None, fontsize=8):
    """A horizontal reference line -- boost -- and its label.

    The label is parked at the dial position where the curve is FURTHEST from the
    line, on the far side of it. Fixed at a third of the way across -- the previous
    rule -- it lands on the curve about half the time, and a label sitting on the
    line it names is the one thing this panel cannot afford.
    """
    ax.axhline(v, color=color, lw=1.4, ls=(0, (5, 3)), zorder=2)
    if xs is None or not len(xs):
        ax.annotate(text, xy=(0.30, v), xycoords=("axes fraction", "data"),
                    xytext=(0, 4), textcoords="offset points",
                    fontsize=fontsize, color=color)
        return
    xs, ys = np.asarray(xs, float), np.asarray(ys, float)
    i = int(np.nanargmax(np.abs(ys - v)))
    higher = ys[i] < v                       # the curve sits at a LOWER value there,
    if ax.yaxis_inverted():                  # which on a place axis is further UP
        higher = not higher
    # ... unless that side is the edge of the panel, where the label would be cut off
    edge = float(ax.transLimits.transform((0, v))[1])
    if higher and edge > 0.92:
        higher = False
    elif not higher and edge < 0.08:
        higher = True
    va, dy = ("bottom", 4) if higher else ("top", -4)
    ax.annotate(text, xy=(float(np.clip(xs[i], 0.15, 0.85)), v), xycoords="data",
                xytext=(0, dy), textcoords="offset points", fontsize=fontsize,
                color=color, va=va, ha="center")

def alpha_axis(ax, bottom=True):
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.grid(axis="y", color=C["grid"], lw=0.8)
    ax.set_axisbelow(True)
    if bottom:
        ax.set_xlabel("alpha    0 = identity alone  ->  1 = ESM nodes")

def finish(fig, title=None, legend_ax=None, ncol=4, size=12):
    """One layout pass for the whole figure: pack the panels, then reserve a strip at
    the top for the title and one at the bottom for the legend.

    In that order, and in one place, because the alternative drifts: a suptitle parked
    at y=1.002 clears two rows of panels and sits on top of four, and a tight_layout
    called after the reservation silently takes it back.
    """
    fig.tight_layout()
    H = fig.get_figheight()
    h, l = legend_ax.get_legend_handles_labels() if legend_ax is not None else ([], [])
    fig.subplots_adjust(top=fig.subplotpars.top - (0.40 / H if title else 0),
                        bottom=fig.subplotpars.bottom + (0.55 / H if h else 0))
    if title:
        fig.suptitle(title, fontsize=size, y=1 - 0.08 / H, va="top")
    if h:
        fig.legend(h, l, loc="lower center", ncol=ncol, frameon=False, fontsize=9)

def note_empty(ax, msg="not run yet"):
    ax.text(0.5, 0.5, msg, transform=ax.transAxes, ha="center", va="center",
            color=C["muted"], fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

def savefig(fig, name):
    if not SAVE_FIGS:
        return
    d = pathlib.Path(FIG_DIR); d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / f"{name}.png", dpi=160, bbox_inches="tight")
    print(f"  -> {d / (name + '.png')}")

---
# 1 · Geometry: does the cloud travel between the two ends?

Two reference clouds, both fit on **train rows only**:

- **ESM** — the raw protein embedding. Structure.
- **response profile** — the train receptor × train-odorant response matrix, row-centred.
  Function. No test odorant enters it; a diagnostic that peeked would report the leak as
  a success.

Three second-order measures, all reported so that **larger is more aligned** (Procrustes
as 1 − disparity): RSA compares neighbour orderings, CCA the shared linear subspace,
Procrustes the shape after the best rigid fit.

The dial working means `vs ESM` **rises** with alpha and `vs response` falls — the v9
direction, opposite to v8.

> On **M2OR read the `fun` column as an upper bound**. That matrix is sparse, so most
> cells are "not assayed" rather than "no response", and the imputation writes the global
> mean into them. What survives is partly *which pairs were measured* — assay design, not
> binding. The `tested mask` control put that share at 76–103% of the graph's own gain on
> M2OR. On the complete insect panels it does not arise.

In [ ]:
geo = ag.geometry(test, scale=GEOM_SCALE)
fig, AX = panels(len(SERIES), len(ag.GEOMS), h=2.6)
for i, s in enumerate(SERIES):
    for j, g in enumerate(ag.GEOMS):
        ax = AX[i][j]
        got = False
        for ref in ag.REFS:
            d = geo[(geo.series == s) & (geo.geom == g) & (geo.ref == ref)]
            if not d.empty:
                got = True
                band(ax, d, C[ref], label=ag.REF_LABEL[ref] if i == j == 0 else None)
        if not got:
            note_empty(ax)
            continue
        alpha_axis(ax, bottom=(i == len(SERIES) - 1))
        if i == 0:
            ax.set_title(ag.GEOM_LABEL[g], fontsize=10)
        if j == 0:
            ax.set_ylabel(s, fontsize=9)
finish(fig, f"1.1  alignment of the receptor cloud with each end  [{GEOM_SCALE}]",
       legend_ax=AX[0][0], ncol=2)
savefig(fig, "1_1_geometry_curves")

In [ ]:
# 1.2  where the two alignments cross, per (series, measure). Each reference curve is
# min-max scaled inside its own panel first -- raw RSA against ESM and against a
# response matrix are not on one scale, so their literal intersection would be an
# artefact of that.
cx = ag.crossover(geo)
if cx.empty or cx["alpha_cross"].isna().all():
    print("no crossing on any panel: the two alignments never swap order over the dial "
          "-- which is itself the finding, not a failure")
else:
    fig, ax = panels(1, 1, w=7, h=0.5 + 0.42 * len(SERIES))
    ax = ax[0][0]
    ys = {s: k for k, s in enumerate(SERIES)}
    for g, mk in zip(ag.GEOMS, ["o", "s", "D"]):
        d = cx[cx.geom == g].dropna(subset=["alpha_cross"])
        ax.plot(d["alpha_cross"], [ys[s] for s in d.series], mk, ms=8, mec="white",
                mew=1.2, label=ag.GEOM_LABEL[g], color=C["dial"], alpha=0.75)
    ax.set_yticks(range(len(SERIES))); ax.set_yticklabels(SERIES, fontsize=9)
    ax.set_xlim(-0.02, 1.02); ax.set_xlabel("alpha at which structure overtakes function")
    ax.grid(axis="x", color=C["grid"], lw=0.8); ax.set_axisbelow(True)
    ax.set_title("1.2  the crossover", fontsize=12, loc="left")
    ax.legend(frameon=False, fontsize=8, ncol=3, loc="upper center",
              bbox_to_anchor=(0.5, -0.35))
    savefig(fig, "1_2_crossover")
display(cx.round(3))

In [ ]:
# 1.3  how far the cloud actually travels. Per (series, measure, reference): the value
# at each END of the dial, as a dumbbell. A short bar means the geometry does not move
# -- and then no performance difference along the dial can be attributed to it.
rows = []
for (s, g, r), d in geo.groupby(["series", "geom", "ref"]):
    d = d.sort_values("alpha")
    if len(d) < 2:
        continue
    rows.append(dict(series=s, geom=g, ref=r, a0=float(d["mean"].iloc[0]),
                     a1=float(d["mean"].iloc[-1]),
                     travel=float(d["mean"].iloc[-1] - d["mean"].iloc[0])))
trav = pd.DataFrame(rows)
if trav.empty:
    print("no geometry to travel")
else:
    fig, AX = panels(1, len(ag.GEOMS), w=3.6, h=0.6 + 0.42 * len(SERIES))
    for j, g in enumerate(ag.GEOMS):
        ax = AX[0][j]
        d = trav[trav.geom == g]
        lab, y = [], 0
        for s in SERIES:
            for r in ag.REFS:
                q = d[(d.series == s) & (d.ref == r)]
                if q.empty:
                    continue
                a0, a1 = float(q.a0.iloc[0]), float(q.a1.iloc[0])
                ax.plot([a0, a1], [y, y], color=C[r], lw=2, alpha=0.55, zorder=2)
                ax.plot([a0], [y], "o", color=C[r], ms=6, mec="white", mew=1, zorder=3)
                ax.plot([a1], [y], "D", color=C[r], ms=6, mec="white", mew=1, zorder=3)
                lab.append(f"{s.split(' / ')[0]}/{s.split(' / ')[-1][:5]} {r}")
                y += 1
        ax.set_yticks(range(len(lab))); ax.set_yticklabels(lab, fontsize=7)
        ax.set_title(ag.GEOM_LABEL[g], fontsize=9)
        ax.grid(axis="x", color=C["grid"], lw=0.8); ax.set_axisbelow(True)
        ax.invert_yaxis()
    finish(fig, "1.3  travel from alpha=0 (circle) to alpha=1 (diamond)")
    savefig(fig, "1_3_travel")
    display(trav.pivot_table(index=["series", "ref"], columns="geom",
                             values="travel").round(3))

---
# 2 · Performance along the dial

## How every number below is reduced, and why in that order

1. **Pair on `(fold, seed)`.** Two arms of one sweep ran on the same held-out rows with
   the same model draw, so differencing removes both nuisances at once.
2. **Average the model seeds inside each fold.** What is left is one number per held-out
   set: the graph's initialisation, the bag and the head's subsample averaged away, the
   fold's own character untouched.
3. **Take the interval over the folds.** Those five numbers are the independent sample.

Step 2 is not cosmetic. Five seeds on one fold rest on the *same rows* — they are one
observation about generalisation, not five. Counting the 25 `(fold, seed)` cells of a
five-seed grid as 25 observations puts `t(24)=2.06` and `sqrt(25)` into an interval whose
effective sample size is 5, which makes it roughly **half its honest width**, and every
gap that then reads as significant inherits the error.

Averaging still buys the seeds' worth: each fold mean carries a fifth of the seed
variance, so the between-fold spread shrinks toward the pure fold-difficulty component.
The narrowing arrives through the numerator, which is real, rather than through the
denominator, which would be invented. **2.6 shows both spreads side by side.**

## Why the advantage and the place carry the story

A raw metric along the dial mixes two things: how good the model is and how hard the
folds were. The **advantage over boost** removes the second by differencing inside each
fold, and the **place** removes it by ranking inside each fold. They are the two honest
summaries, and each is shown twice — once averaged, once **per fold**, because an
advantage of +0.01 that holds on five folds and one that is +0.06 on one fold and zero on
four are the same mean and are not the same result.

**The place here is a duel, not a league table.** Each dial position is ranked against
one opponent — the boosting baseline — on each split separately, so a split scores 1 if
the model is ahead on it and 2 if it is behind. Averaged over `n` splits the place lands
on a **1…2 scale in steps of 1/n**: with five folds, 1.0 · 1.2 · 1.4 · 1.6 · 1.8 · 2.0,
where 1.0 is ahead everywhere, 2.0 behind everywhere and 1.5 an even division.

That scale is readable in a way the k-way race is not. In a league of every alpha plus
the reference arms, a position can slip from 3rd to 4th because two *other* positions
moved, while its own verdict against boost never changed. The duel cannot do that: it
only ever reports the comparison the story is about. What it gives up is magnitude — a
split won by 0.001 and one won by 0.1 both score 1 — which is exactly what the advantage
panels above carry, and why the two are read together.

In [ ]:
# 2.1  the metric of record, with boost as the reference line
fig, G = grid(len(SERIES), COLS_SERIES, h=2.9)
for (ax, bot, left), s in zip(G, SERIES):
    ds = test[test.series == s].dataset.iloc[0]
    m = MET[ds]
    cur = ag.curve(test[test.series == s], m, level=CI_LEVEL).sort_values("alpha")
    band(ax, cur, C["dial"])
    lv = ag.levels(test[test.series == s], m, level=CI_LEVEL)
    q = lv[lv.arm == "boost_full"]
    if not q.empty:
        ref_line(ax, float(q["mean"].iloc[0]), C["boost_full"],
                 ag.ARM_LABEL["boost_full"], cur["alpha"], cur["mean"])
    # the metric of record differs BETWEEN panels here (R2 on the insects, AUROC on
    # the binary pool), so the y label stays on every one of them
    ax.set_title(s, fontsize=10, loc="left"); ax.set_ylabel(m)
    alpha_axis(ax, bottom=bot); floor_bar(ax, s)
finish(fig, "2.1  metric of record along the dial   (interval over FOLDS, seeds "
            "averaged within each)")
savefig(fig, "2_1_metric_of_record")

In [ ]:
# 2.2  THE ADVANTAGE OVER BOOST, with every fold drawn. The line is the mean of the
# points beneath it; the band is the t interval over exactly those points. A dial
# position is a win only where the points agree, and this panel is where that shows.
fig, G = grid(len(SERIES), COLS_SERIES, h=3.1)
for (ax, bot, left), s in zip(G, SERIES):
    sub = test[test.series == s]
    m = MET[sub.dataset.iloc[0]]
    d = ag.delta_vs(sub, m, level=CI_LEVEL)
    if d.empty:
        note_empty(ax); ax.set_title(s, fontsize=10, loc="left"); continue
    pf = ag.per_fold_delta(sub, m)
    for f, g in pf.groupby("fold"):
        g = g.sort_values("alpha")
        ax.plot(g["alpha"], g["d"], color=C["dial"], lw=0.9, alpha=0.30, zorder=2)
        ax.plot(g["alpha"], g["d"], "o", color=C["dial"], ms=3, alpha=0.45, zorder=2)
    band(ax, d, C["dial"])
    ax.axhline(0, color=C["boost_full"], lw=1.4, ls=(0, (5, 3)), zorder=1)
    won = d.set_index("alpha")["won"]
    best = d.loc[d["mean"].idxmax()]
    ax.annotate(f"best {best['mean']:+.3f} at a={best['alpha']:g}"
                f"   ({int(best['won'])}/{int(best['n'])} folds)",
                xy=(0.03, 0.94), xycoords="axes fraction", fontsize=8,
                color=C["muted"], va="top")
    ax.set_title(s, fontsize=10, loc="left")
    ax.set_ylabel(f"{m}   (dial - boost, paired)")
    alpha_axis(ax, bottom=bot)
finish(fig, "2.2  advantage over boost -- thin lines are the folds, thick is "
            "their mean")
savefig(fig, "2_2_advantage_per_fold")

In [ ]:
# 2.3  the advantage on EVERY metric, not just the one of record. A win that appears
# only on the headline metric is a different claim from one that appears on all four.
# METRICS="headline" keeps this to five panels on a regression cell; "all" opens the
# full battery, which is twelve and wraps onto four rows.
for s in SERIES:
    sub = test[test.series == s]
    ds = sub.dataset.iloc[0]
    mets = metric_list(test, ds)
    fig, G = grid(len(mets), COLS_METRIC, w=3.3, h=2.5)
    for (ax, bot, left), mm in zip(G, mets):
        d = ag.delta_vs(sub, mm, level=CI_LEVEL)
        if d.empty:
            note_empty(ax); ax.set_title(mm, fontsize=9); continue
        band(ax, d, C["dial"])
        ax.axhline(0, color=C["boost_full"], lw=1.3, ls=(0, (5, 3)))
        lower = mm in ag.LOWER_IS_BETTER
        ax.set_title(mm + ("  (below 0 is the win)" if lower else ""), fontsize=9)
        ax.set_ylabel("dial - boost, paired" if left else "")
        alpha_axis(ax, bottom=bot)
    finish(fig, f"2.3  {s} -- advantage over boost, every metric", size=11)
    savefig(fig, f"2_3_advantage_{s.replace(' / ', '_').replace(' ', '')}")

In [ ]:
# 2.4  THE PLACE AGAINST BOOST, per metric. Two models, ranked on each split on its
# own: 1 where the dial is ahead, 2 where boost is. Averaged over n splits the place
# moves in steps of 1/n -- with five folds 1.0 / 1.2 / ... / 2.0 -- so the number says
# how OFTEN the model wins, and its distance from 1.5 says how consistently.
for s in SERIES:
    sub = test[test.series == s]
    ds = sub.dataset.iloc[0]
    mets = SHOW_PLACES_FOR or metric_list(test, ds)
    fig, G = grid(len(mets), COLS_METRIC, w=3.3, h=2.5)
    for (ax, bot, left), mm in zip(G, mets):
        d = ag.duel(sub, mm).sort_values("alpha")
        if d.empty:
            note_empty(ax); ax.set_title(mm, fontsize=9); continue
        nf = int(d["n"].max())
        ax.plot(d["alpha"], d["place"], color=C["dial"], lw=2, zorder=3)
        ax.plot(d["alpha"], d["place"], "o", color=C["dial"], ms=4,
                mec="white", mew=1, zorder=4)
        ax.set_ylim(2.06, 0.94)      # the whole scale, always: a curve auto-scaled to
                                     # 1.0-1.2 looks like a landslide and is one fold
        step = 1 / nf if 0 < nf <= 10 else 0.25          # one tick per split
        ax.set_yticks(np.round(np.arange(1, 2 + 1e-9, step), 3))
        ax.axhline(1.5, color=C["muted"], lw=1, ls=":", zorder=1)
        ax.annotate("even split", xy=(0.99, 1.5), xycoords=("axes fraction", "data"),
                    xytext=(0, 3), textcoords="offset points", ha="right",
                    fontsize=7, color=C["muted"])
        ax.set_title(f"{mm}   (vs boost, {nf} splits)", fontsize=9)
        ax.set_ylabel("place of 2" if left else "")
        alpha_axis(ax, bottom=bot)
    finish(fig, f"2.4  {s} -- place against boost, split by split "
                f"(1 = ahead everywhere, 2 = behind everywhere)", size=11)
    savefig(fig, f"2_4_places_{s.replace(' / ', '_').replace(' ', '')}")

In [ ]:
# 2.5  THE DUEL, SPLIT BY SPLIT. What the mean in 2.4 averages away: a place of 1.4 is
# three splits won and two lost, and WHICH two matters -- if every dial position loses
# the same fold, that fold is hard, not the dial's problem.
for s in SERIES:
    sub = test[test.series == s]
    m = MET[sub.dataset.iloc[0]]
    rows = ag.duel_matrix(sub, m)
    if rows.empty:
        continue
    rows = rows.assign(who=[f"a={a:g}" for a in rows["alpha"]])
    if PLACE_ALPHAS is not None:
        rows = rows[~rows.who.str.startswith("a=")
                    | rows.alpha.isin(PLACE_ALPHAS)]
    M = rows.pivot_table(index="who", columns="fold", values="place")
    D = rows.pivot_table(index="who", columns="fold", values="d")
    order = M.mean(axis=1).sort_values().index
    M, D = M.loc[order], D.loc[order]
    fig, ax = panels(1, 1, w=2.2 + 0.95 * M.shape[1], h=1.0 + 0.36 * M.shape[0])
    ax = ax[0][0]
    # two states, so two colours and no ramp: ahead of boost, or behind it
    ax.imshow(M.to_numpy(), cmap=mcolors.ListedColormap([C["dial"], C["fun"]]),
              aspect="auto", vmin=1, vmax=2)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v, dd = M.to_numpy()[i, j], D.to_numpy()[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{int(v)}{chr(10)}{dd:+.3f}", ha="center", va="center",
                        fontsize=7, color="white", linespacing=1.3)
        ax.text(M.shape[1] - 0.4, i, f"  {M.iloc[i].mean():.2f}", ha="left",
                va="center", fontsize=9, color=C["ink"], fontweight="bold")
    # the mean column gets its name from the tick row, not a floating label that would
    # sit in the title
    ax.set_xticks(list(range(M.shape[1])) + [M.shape[1] + 0.05])
    ax.set_xticklabels([f"f{c}" for c in M.columns] + ["mean"])
    ax.set_yticks(range(M.shape[0])); ax.set_yticklabels(M.index, fontsize=8)
    ax.set_xlim(-0.5, M.shape[1] + 0.3)          # room for the mean column
    ax.set_title(f"2.5  {s} -- place against boost on {m}, split by split  "
                 f"(1 = ahead, 2 = behind; the gap is under it; mean at the right)",
                 fontsize=10, loc="left")
    ax.set_xlabel("held-out fold")
    fig.tight_layout()
    savefig(fig, f"2_5_place_matrix_{s.replace(' / ', '_').replace(' ', '')}")

In [ ]:
# 2.6  WAS THE SEED AVERAGING WORTH IT, AND WHAT IS LEFT. Two spreads, two questions:
#   within   how much the same model on the same rows moves between initialisations
#   between  how much the answer moves between held-out sets -- what the interval is on
# `floor` is within/sqrt(n_seeds): the model noise still inside `between` after
# averaging. Where `between` sits on the floor, the folds agree and the residual is
# model noise; where it towers over it, the folds genuinely disagree and no number of
# seeds will help.
fig, G = grid(len(SERIES), COLS_SERIES, h=2.9)
for (ax, bot, left), s in zip(G, SERIES):
    sub = test[test.series == s]
    m = MET[sub.dataset.iloc[0]]
    ns = ag.noise_split(sub, m).sort_values("alpha")
    if ns.empty or not np.isfinite(ns["within"]).any():
        note_empty(ax, "one model seed -- nothing to decompose")
        ax.set_title(s, fontsize=10, loc="left"); continue
    ax.plot(ns["alpha"], ns["between"], color=C["dial"], lw=2,
            label="between folds (what the interval is on)")
    ax.plot(ns["alpha"], ns["within"], color=C["val"], lw=2,
            label="within a fold (model seeds)")
    ax.fill_between(ns["alpha"], 0, ns["floor"], color=C["val"], alpha=0.15, lw=0,
                    label="model noise left after averaging")
    ax.set_title(f"{s}   ({ns['n_seeds'].mean():.0f} seeds x "
                 f"{int(ns['n_folds'].max())} folds)", fontsize=10, loc="left")
    ax.set_ylabel(f"SD of {m}")
    ax.set_ylim(bottom=0)
    alpha_axis(ax, bottom=bot)
finish(fig, "2.6  model noise vs split individuality", legend_ax=G[0][0], ncol=3)
savefig(fig, "2_6_noise_split")

In [ ]:
# 2.7  the numbers behind 2.1-2.6
for s in SERIES:
    sub = test[test.series == s]
    m = MET[sub.dataset.iloc[0]]
    cur = ag.curve(sub, m, level=CI_LEVEL)[["alpha", "mean", "hw", "n", "seeds"]]
    dlt = ag.delta_vs(sub, m, level=CI_LEVEL)[["alpha", "mean", "hw", "won", "n"]]
    duel = ag.duel(sub, m)[["alpha", "place", "wins", "n"]]
    duel = duel.rename(columns={"place": "place vs boost", "n": "splits"})
    league = ag.places(sub, m)
    league = (league[league.alpha.notna()][["alpha", "place", "k"]]
              .rename(columns={"place": "place of k"}))
    out = (cur.rename(columns={"mean": m, "hw": "+/-", "n": "folds"})
           .merge(dlt.rename(columns={"mean": "d vs boost", "hw": "d +/-",
                                      "n": "paired folds"}), on="alpha", how="left")
           .merge(duel, on="alpha", how="left")
           .merge(league, on="alpha", how="left"))
    print(f"{chr(10)}=== {s}   [{m}]   repeat floor "
          f"{FLOOR.get(s, float('nan')):.4f}   (interval over FOLDS; `seeds` is how "
          f"many draws each fold mean averages)")
    display(out.round(4))

---
# 3 · Choosing alpha — on validation, and only then on test

The rules this block enforces:

1. **The search runs on validation rows.** They were never used to train the graph
   (message-passing edges come from train only, the epoch count is fixed, val is never
   consulted) and never used to fit the head. They are also the same *shape* as test —
   `inductive_molecule_v5` draws val molecules disjoint from train and test, and
   `our_inductive` spreads val over the same dynamic-range order — so a cold-molecule
   choice is made on cold-molecule evidence.
2. **The 1-SE rule.** Alphas within one standard error of the leader are
   indistinguishable from it. An argmax out of a flat set is noise-chasing.
3. **Test is read once**, at the chosen alpha, and the gap to the best-on-test alpha is
   printed as the *optimism avoided* — exactly what choosing on test would have added to
   the claim.

In [ ]:
import argparse
if val is None:
    print("no validation frame -- see the load cell for the command that builds it")
else:
    ARGS = argparse.Namespace(metric=None, loo=False, select_on="val", at_alpha=None)
    metric_of = ac._metric_of(val, ARGS)
    sc_val = ac.scores(val, metric_of)
    rk_val = ac.ranked(sc_val)
    tied, cut = ac.one_se(rk_val)
    best, _ = ac.chosen_alpha(val, metric_of)
    display(rk_val.round(3))
    print(f"\nBEST on validation: alpha = {best:g}   (mean rank "
          f"{float(rk_val.iloc[0].mean_rank):.2f})")
    print(f"WITHIN 1 SE of it (<= {cut:.2f}): "
          + ", ".join(f"{a:g}" for a in sorted(tied.alpha.dropna())))
    if len(tied) > 1:
        print("   -> the dial is FLAT there. Fix alpha at the end you can argue for and "
              "say so;\n      this table is the evidence that nothing was lost by it.")

In [ ]:
# 3.2  the validation leaderboard as a curve. The 1-SE band is the honest width of the
# choice: everything inside it is one answer, not a ranking.
if val is not None:
    gate = rk_val[rk_val.alpha.notna()].sort_values("alpha")
    fig, ax = panels(1, 1, w=7.2, h=3.2)
    ax = ax[0][0]
    ax.axhspan(float(rk_val[rk_val.alpha.notna()].mean_rank.min()), cut,
               color=C["dial"], alpha=0.10, lw=0, label="within 1 SE of the best")
    ax.errorbar(gate["alpha"], gate["mean_rank"], yerr=gate["se"], fmt="none",
                ecolor=C["dial"], alpha=0.5, elinewidth=1.2, capsize=3)
    ax.plot(gate["alpha"], gate["mean_rank"], color=C["dial"], lw=2, zorder=3)
    ax.plot(gate["alpha"], gate["mean_rank"], "o", color=C["dial"], ms=5,
            mec="white", mew=1, zorder=4)
    ax.invert_yaxis()                       # 1 = best. Before the labels: they read the
                                            # axis direction, and an anchor in DATA
                                            # coordinates flips with it
    q = rk_val[rk_val.competitor == "boost"]
    if not q.empty:
        ref_line(ax, float(q.mean_rank.iloc[0]), C["boost_full"], "boost",
                 gate["alpha"], gate["mean_rank"])
    ax.axvline(best, color=C["val"], lw=1.4)
    # the top of the AXES, not the top value: on an inverted axis those are opposite
    # ends, and this label used to land in the x tick labels
    ax.annotate(f"chosen  alpha={best:g}", xy=(best, 1.0),
                xycoords=("data", "axes fraction"), xytext=(4, -12),
                textcoords="offset points", color=C["val"], fontsize=9, va="top")
    ax.set_ylabel("mean rank across cells (1 = best)")
    alpha_axis(ax)
    ax.set_title("3.2  the choice, on VALIDATION", fontsize=12, loc="left")
    ax.legend(frameon=False, fontsize=8)
    savefig(fig, "3_2_val_leaderboard")

In [ ]:
# 3.3  val against test, curve by curve. Not a check of the choice -- it is a check of
# whether validation is a usable GUIDE here at all: if the two disagree about the shape,
# the dial is inside the noise on this cell and no split can choose within it.
if val is not None:
    fig, G = grid(len(SERIES), COLS_SERIES, h=2.9)
    for (ax, bot, left), s in zip(G, SERIES):
        ds = test[test.series == s].dataset.iloc[0]
        m = MET[ds]
        first = ax is G[0][0]
        band(ax, ag.curve(test[test.series == s], m, level=CI_LEVEL), C["dial"],
             label="test" if first else None)
        vs = val[val.series == s]
        if not vs.empty:
            band(ax, ag.curve(vs, m, level=CI_LEVEL), C["val"],
                 label="validation" if first else None)
        ax.axvline(best, color=C["muted"], lw=1, ls=":")
        ax.set_title(s, fontsize=10, loc="left"); ax.set_ylabel(m)
        alpha_axis(ax, bottom=bot)
    finish(fig, "3.3  validation vs test -- dotted line is the chosen alpha",
           legend_ax=G[0][0], ncol=2)
    savefig(fig, "3_3_val_vs_test")

In [ ]:
# 3.4  THE ONLY PLACE TEST IS READ. Per cell: the value at the chosen alpha, its paired
# advantage over boost, and how far the luckiest alpha on these very folds would have
# outscored it -- the optimism that choosing on test would have quietly added.
if val is not None:
    ac.confirm(val, test, argparse.Namespace(metric=None, at_alpha=None))

---
## What can be written down from this

Fill these in from the blocks above; each is a sentence the notebook has evidence for.

- **Does the cloud move?** 1.3 gives the travel per measure and reference. If it is
  short, the dial changes the input without changing the receptor geometry, and no
  performance difference along it can be attributed to structure.
- **Does the score move?** 2.1 against the repeat bar, and 2.2 for the paired version.
  A difference smaller than the bar is not resolvable at one model seed.
- **Which alpha, and does it matter?** 3.1 gives the leader and the 1-SE set. If the set
  is wide, the honest sentence is *"alpha was fixed at the end of the dial on the
  argument, not tuned; the sweep is the evidence that nothing was lost"*.
- **What does it cost to be honest?** The `best there` column in 3.4. That number is the
  margin a test-chosen alpha would have claimed and this protocol gives up.

Anything else — three datasets against each other, the molecule sources side by side —
belongs to `paper_tables.py`, which is the scoreboard of record. This notebook is the
shape of one dial.